In [ ]:
from pathlib import Path
import subprocess,sys,pandas as pd
repo=Path('/tmp/PCC'); subprocess.run(['git','clone','--quiet','https://github.com/changxinjiresearch/PCC.git',str(repo)],check=True); subprocess.run(['git','-C',str(repo),'checkout','--quiet','ba48846'],check=True)
root=Path('/kaggle/input'); candidates=[p.parent for p in root.rglob('ALL_CASE_METHOD_METRICS.csv') if (p.parent/'held_out_p0').exists()]; assert candidates; frozen=candidates[0]
manifest=pd.read_csv(frozen/'LOCKED_CASE_MANIFEST.csv'); manifest[['current_t1c_path','current_mask_path','future_mask_path']]=manifest[['current_t1c_path','current_mask_path','future_mask_path']].replace('/kaggle/input/datasets/stacyvangepuram/mu-glioma-post','/kaggle/input/mu-glioma-post',regex=True); alias=Path('/kaggle/working/frozen_alias'); alias.mkdir(); (alias/'held_out_p0').symlink_to(frozen/'held_out_p0',target_is_directory=True); (alias/'retrospective').symlink_to(frozen/'retrospective',target_is_directory=True); manifest.to_csv(alias/'LOCKED_CASE_MANIFEST.csv',index=False)
repeat_files=list(root.rglob('IMPERFECT_GUIDANCE_REPEAT_METRICS.csv')); assert len(repeat_files)==2; repeats=pd.concat([pd.read_csv(p) for p in repeat_files],ignore_index=True); repeat_path=Path('/kaggle/working/original_repeats.csv'); repeats.to_csv(repeat_path,index=False); assert len(repeats)==5160
out=Path('/kaggle/working/pcc_internal_validity_patch_2026'); subprocess.run([sys.executable,'-m','experiments.run_internal_validity_patch','--family','no_smoothing','--frozen-root',str(alias),'--original-repeats',str(repeat_path),'--output-root',str(out),'--shard-index','0','--shard-count','2'],cwd=repo,check=True)
